# ⏱️ Rata-Rata Waktu Agent

Notebook ini menghitung rata-rata waktu eksekusi untuk ketiga agent dalam pipeline pengujian:
- **Profile Agent** — waktu generate profil NPC
- **Dialogue Agent** — waktu generate dialog
- **Critic Agent** — waktu evaluasi per iterasi dialog-critic

In [4]:
import pandas as pd
import numpy as np
import os

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".", "output")

# Baca ketiga CSV
df_profile = pd.read_csv(os.path.join(OUTPUT_DIR, "profile_testing.csv"))
df_dialogue = pd.read_csv(os.path.join(OUTPUT_DIR, "dialogue_testing.csv"))
df_critic = pd.read_csv(os.path.join(OUTPUT_DIR, "critic_testing.csv"))

print("=" * 60)
print("📊 DATA LOADED")
print("=" * 60)
print(f"Profile testing : {len(df_profile)} baris, kolom: {list(df_profile.columns)}")
print(f"Dialogue testing: {len(df_dialogue)} baris, kolom: {list(df_dialogue.columns)}")
print(f"Critic testing  : {len(df_critic)} baris, kolom: {list(df_critic.columns)}")
print()

# Cek missing values
print("Missing values:")
print(f"  profile  : {df_profile['waktu_detik'].isna().sum()}")
print(f"  dialogue : {df_dialogue['waktu_detik'].isna().sum()}")
print(f"  critic   : {df_critic['waktu_detik'].isna().sum()}")

📊 DATA LOADED
Profile testing : 60 baris, kolom: ['jenis_profile', 'nama', 'usia', 'gender', 'background', 'masalah_hari_ini', 'ocean_O', 'ocean_C', 'ocean_E', 'ocean_A', 'ocean_N', 'max_fails', 'reaksi_gaya', 'waktu_detik']
Dialogue testing: 420 baris, kolom: ['jenis_profile', 'nama_profile', 'jenis_dialog', 'teks_dialog', 'pilihan_jawaban', 'minuman_dipesan', 'jawaban_dipilih', 'waktu_detik']
Critic testing  : 526 baris, kolom: ['jenis_profile', 'nama_profile', 'iterasi_ke', 'jenis_dialog', 'teks_dialog', 'teks_kritik', 'skor_kritik', 'waktu_detik']

Missing values:
  profile  : 0
  dialogue : 0
  critic   : 0


## 1. Profile Agent — Waktu Generate Profil

Rata-rata waktu yang dibutuhkan Profile Agent untuk menghasilkan satu profil NPC (OCEAN + background + masalah).

In [5]:
# ── Overall ──
w = df_profile["waktu_detik"]
print("─ Profile Agent ──────────────────────────────────────")
print(f"  Mean   : {w.mean():.2f} detik")
print(f"  Std    : {w.std():.2f} detik")
print(f"  Min    : {w.min():.2f} detik")
print(f"  Max    : {w.max():.2f} detik")
print(f"  Total  : {w.sum():.2f} detik ({w.sum()/60:.2f} menit)")
print()

# ── Per jenis_profile ──
print("─ Per Jenis Profile ──────────────────────────────────")
profile_group = df_profile.groupby("jenis_profile")["waktu_detik"].agg(["mean", "std", "min", "max", "count"]).round(2)
profile_group = profile_group.sort_values("mean", ascending=False)
print(profile_group.to_string())

─ Profile Agent ──────────────────────────────────────
  Mean   : 5.49 detik
  Std    : 0.86 detik
  Min    : 4.09 detik
  Max    : 7.99 detik
  Total  : 329.39 detik (5.49 menit)

─ Per Jenis Profile ──────────────────────────────────
                  mean   std   min   max  count
jenis_profile                                  
orang tua wanita  6.15  0.89  4.96  7.55     10
remaja pria       5.66  0.79  4.87  7.07     10
orang tua pria    5.58  1.05  4.77  7.99     10
dewasa pria       5.41  0.41  4.71  6.12     10
dewasa wanita     5.07  0.82  4.09  6.38     10
remaja wanita     5.07  0.70  4.19  6.46     10


## 2. Dialogue Agent — Waktu Generate Dialog

Rata-rata waktu yang dibutuhkan Dialogue Agent untuk menghasilkan satu dialog (dari 7 jenis: pesanan, pesanan_salah, marah, berhasil, curhat, reaksi, closing).

In [6]:
# ── Overall ──
w = df_dialogue["waktu_detik"]
print("─ Dialogue Agent ─────────────────────────────────────")
print(f"  Mean   : {w.mean():.2f} detik")
print(f"  Std    : {w.std():.2f} detik")
print(f"  Min    : {w.min():.2f} detik")
print(f"  Max    : {w.max():.2f} detik")
print(f"  Total  : {w.sum():.2f} detik ({w.sum()/60:.2f} menit)")
print()

# ── Per jenis_profile ──
print("─ Per Jenis Profile ──────────────────────────────────")
dg_profile = df_dialogue.groupby("jenis_profile")["waktu_detik"].agg(["mean", "std", "min", "max", "count"]).round(2)
dg_profile = dg_profile.sort_values("mean", ascending=False)
print(dg_profile.to_string())
print()

# ── Per jenis_dialog ──
print("─ Per Jenis Dialog ───────────────────────────────────")
dg_type = df_dialogue.groupby("jenis_dialog")["waktu_detik"].agg(["mean", "std", "min", "max", "count"]).round(2)
# Urut sesuai flow
dialog_order = ["pesanan", "pesanan_salah", "marah", "berhasil", "curhat", "reaksi", "closing"]
dg_type = dg_type.reindex(dialog_order)
print(dg_type.to_string())

─ Dialogue Agent ─────────────────────────────────────
  Mean   : 2.42 detik
  Std    : 1.15 detik
  Min    : 1.02 detik
  Max    : 18.13 detik
  Total  : 1017.49 detik (16.96 menit)

─ Per Jenis Profile ──────────────────────────────────
                  mean   std   min    max  count
jenis_profile                                   
orang tua wanita  2.84  2.14  1.11  18.13     70
dewasa wanita     2.44  0.85  1.02   4.51     70
orang tua pria    2.44  0.81  1.14   4.68     70
remaja pria       2.30  0.79  1.07   4.50     70
remaja wanita     2.30  0.80  1.16   4.82     70
dewasa pria       2.22  0.72  1.16   4.00     70

─ Per Jenis Dialog ───────────────────────────────────
               mean   std   min    max  count
jenis_dialog                                 
pesanan        1.64  0.32  1.07   2.24     60
pesanan_salah  2.40  0.58  1.02   3.85     60
marah          2.04  0.51  1.16   3.41     60
berhasil       1.76  0.37  1.15   2.86     60
curhat         3.75  0.58  2.65   5.4

## 3. Critic Agent — Waktu Evaluasi per Iterasi

Rata-rata waktu yang dibutuhkan Critic Agent untuk mengevaluasi satu iterasi dialog-critic.

In [7]:
# ── Overall ──
w = df_critic["waktu_detik"]
print("─ Critic Agent (per iterasi) ─────────────────────────")
print(f"  Mean   : {w.mean():.2f} detik")
print(f"  Std    : {w.std():.2f} detik")
print(f"  Min    : {w.min():.2f} detik")
print(f"  Max    : {w.max():.2f} detik")
print(f"  Total  : {w.sum():.2f} detik ({w.sum()/60:.2f} menit)")
print()

# ── Per jenis_profile ──
print("─ Per Jenis Profile ──────────────────────────────────")
ct_profile = df_critic.groupby("jenis_profile")["waktu_detik"].agg(["mean", "std", "min", "max", "count"]).round(2)
ct_profile = ct_profile.sort_values("mean", ascending=False)
print(ct_profile.to_string())
print()

# ── Per jenis_dialog ──
print("─ Per Jenis Dialog ───────────────────────────────────")
ct_type = df_critic.groupby("jenis_dialog")["waktu_detik"].agg(["mean", "std", "min", "max", "count"]).round(2)
ct_type = ct_type.reindex(dialog_order)
print(ct_type.to_string())

─ Critic Agent (per iterasi) ─────────────────────────
  Mean   : 8.09 detik
  Std    : 1.82 detik
  Min    : 5.21 detik
  Max    : 33.95 detik
  Total  : 4254.00 detik (70.90 menit)

─ Per Jenis Profile ──────────────────────────────────
                  mean   std   min    max  count
jenis_profile                                   
orang tua pria    8.76  1.43  6.71  15.89    114
orang tua wanita  8.66  1.57  6.09  12.97    106
remaja pria       7.84  3.29  5.21  33.95     74
dewasa pria       7.69  1.26  5.42  12.37     82
dewasa wanita     7.59  1.18  5.75  11.86     75
remaja wanita     7.42  0.98  5.94  10.59     75

─ Per Jenis Dialog ───────────────────────────────────
               mean   std   min    max  count
jenis_dialog                                 
pesanan        7.60  1.37  5.21  11.98     71
pesanan_salah  8.27  1.29  6.11  12.08     80
marah          7.75  0.92  6.00  10.33     66
berhasil       7.32  1.10  5.72  11.89     68
curhat         8.68  1.57  5.94  12.9

## 4. Ringkasan — Tabel Perbandingan 3 Agent

Rata-rata waktu seluruh agent dalam satu tabel.

In [8]:
summary = pd.DataFrame({
    "Agent": ["Profile Agent", "Dialogue Agent", "Critic Agent (per iterasi)"],
    "Mean (detik)": [
        df_profile["waktu_detik"].mean(),
        df_dialogue["waktu_detik"].mean(),
        df_critic["waktu_detik"].mean(),
    ],
    "Std (detik)": [
        df_profile["waktu_detik"].std(),
        df_dialogue["waktu_detik"].std(),
        df_critic["waktu_detik"].std(),
    ],
    "Min (detik)": [
        df_profile["waktu_detik"].min(),
        df_dialogue["waktu_detik"].min(),
        df_critic["waktu_detik"].min(),
    ],
    "Max (detik)": [
        df_profile["waktu_detik"].max(),
        df_dialogue["waktu_detik"].max(),
        df_critic["waktu_detik"].max(),
    ],
    "Total Data": [
        len(df_profile),
        len(df_dialogue),
        len(df_critic),
    ],
})

summary = summary.round(2)
print("=" * 70)
print("  📊 RINGKASAN RATA-RATA WAKTU AGENT")
print("=" * 70)
print(summary.to_string(index=False))
print()

# Total waktu keseluruhan
total_all = df_profile["waktu_detik"].sum() + df_dialogue["waktu_detik"].sum() + df_critic["waktu_detik"].sum()
print(f"Total waktu seluruh agent: {total_all:.2f} detik ({total_all/60:.2f} menit)")

  📊 RINGKASAN RATA-RATA WAKTU AGENT
                     Agent  Mean (detik)  Std (detik)  Min (detik)  Max (detik)  Total Data
             Profile Agent          5.49         0.86         4.09         7.99          60
            Dialogue Agent          2.42         1.15         1.02        18.13         420
Critic Agent (per iterasi)          8.09         1.82         5.21        33.95         526

Total waktu seluruh agent: 5600.87 detik (93.35 menit)


## 5. Dialog-Critic Full Loop — Waktu Total per Sesi Dialog

Waktu total yang dibutuhkan untuk menyelesaikan **seluruh iterasi** (1-10 attempt) dari satu dialog-critic loop. Dihitung dengan menjumlahkan `waktu_detik` semua iterasi untuk kombinasi (nama_profile, jenis_dialog) yang sama.

In [9]:
# Total waktu per dialog-critic loop (jumlah semua iterasi untuk satu dialog)
loop_total = df_critic.groupby(["jenis_profile", "nama_profile", "jenis_dialog"])["waktu_detik"].sum().reset_index()
loop_total.rename(columns={"waktu_detik": "total_waktu_loop"}, inplace=True)

# ── Overall ──
w = loop_total["total_waktu_loop"]
print("─ Dialog-Critic Full Loop (total semua iterasi) ──────")
print(f"  Mean   : {w.mean():.2f} detik")
print(f"  Std    : {w.std():.2f} detik")
print(f"  Min    : {w.min():.2f} detik")
print(f"  Max    : {w.max():.2f} detik")
print(f"  Total  : {w.sum():.2f} detik ({w.sum()/60:.2f} menit)")
print()

# ── Per jenis_dialog ──
print("─ Per Jenis Dialog ───────────────────────────────────")
loop_type = loop_total.groupby("jenis_dialog")["total_waktu_loop"].agg(["mean", "std", "min", "max", "count"]).round(2)
loop_type = loop_type.reindex(dialog_order)
print(loop_type.to_string())
print()

# ── Per jenis_profile ──
print("─ Per Jenis Profile ──────────────────────────────────")
loop_profile = loop_total.groupby("jenis_profile")["total_waktu_loop"].agg(["mean", "std", "min", "max", "count"]).round(2)
loop_profile = loop_profile.sort_values("mean", ascending=False)
print(loop_profile.to_string())

─ Dialog-Critic Full Loop (total semua iterasi) ──────
  Mean   : 12.70 detik
  Std    : 10.58 detik
  Min    : 5.41 detik
  Max    : 110.49 detik
  Total  : 4254.00 detik (70.90 menit)

─ Per Jenis Dialog ───────────────────────────────────
                mean    std   min     max  count
jenis_dialog                                    
pesanan        11.25   8.26  5.41   44.63     48
pesanan_salah  14.07  13.08  6.11   81.99     47
marah          10.66   5.31  6.00   24.71     48
berhasil       10.36   6.52  5.78   33.22     48
curhat         11.76   7.25  5.94   34.95     48
reaksi         18.02  16.80  6.08  110.49     48
closing        12.80  10.52  5.96   67.09     48

─ Per Jenis Profile ──────────────────────────────────
                   mean    std   min     max  count
jenis_profile                                      
orang tua pria    23.78  21.00  6.96  110.49     42
orang tua wanita  18.74   9.49  6.66   35.76     49
dewasa pria       11.26   6.27  6.00   31.92     56
r

## 6. Rata-Rata Iterasi per Dialog

Berapa rata-rata jumlah iterasi (attempt) yang dibutuhkan Critic Agent per dialog sebelum lulus threshold.

In [10]:
# Jumlah iterasi per dialog-critic loop
iter_counts = df_critic.groupby(["jenis_profile", "nama_profile", "jenis_dialog"])["iterasi_ke"].max().reset_index()
iter_counts.rename(columns={"iterasi_ke": "total_iterasi"}, inplace=True)

# ── Overall ──
i = iter_counts["total_iterasi"]
print("─ Rata-Rata Iterasi per Dialog ───────────────────────")
print(f"  Mean   : {i.mean():.2f} iterasi")
print(f"  Std    : {i.std():.2f} iterasi")
print(f"  Min    : {i.min():.0f} iterasi")
print(f"  Max    : {i.max():.0f} iterasi")
print()

# ── Per jenis_dialog ──
print("─ Per Jenis Dialog ───────────────────────────────────")
iter_type = iter_counts.groupby("jenis_dialog")["total_iterasi"].agg(["mean", "std", "min", "max"]).round(2)
iter_type = iter_type.reindex(dialog_order)
print(iter_type.to_string())
print()

# ── Per jenis_profile ──
print("─ Per Jenis Profile ──────────────────────────────────")
iter_profile = iter_counts.groupby("jenis_profile")["total_iterasi"].agg(["mean", "std", "min", "max"]).round(2)
iter_profile = iter_profile.sort_values("mean", ascending=False)
print(iter_profile.to_string())

─ Rata-Rata Iterasi per Dialog ───────────────────────
  Mean   : 1.29 iterasi
  Std    : 0.68 iterasi
  Min    : 1 iterasi
  Max    : 8 iterasi

─ Per Jenis Dialog ───────────────────────────────────
               mean   std  min  max
jenis_dialog                       
pesanan        1.19  0.39    1    2
pesanan_salah  1.38  0.80    1    4
marah          1.12  0.33    1    2
berhasil       1.17  0.43    1    3
curhat         1.10  0.42    1    3
reaksi         1.75  1.21    1    8
closing        1.29  0.50    1    3

─ Per Jenis Profile ──────────────────────────────────
                  mean   std  min  max
jenis_profile                         
orang tua pria    1.79  1.26    1    8
orang tua wanita  1.73  0.86    1    4
dewasa pria       1.21  0.46    1    3
remaja wanita     1.11  0.31    1    2
dewasa wanita     1.07  0.26    1    2
remaja pria       1.06  0.25    1    2
